# 🏠 RoomCanvas AI — Inference Server

**Architecture:** Stable Diffusion 1.5 + ControlNet-MLSD, img2img, batched 3-variation generation  
**Purpose:** Runs as a persistent HTTP inference server inside a Colab T4 GPU runtime, exposed via ngrok.  
**Endpoint:** `POST /infer` accepts a room image + style key and returns 3 redesigned variations + metadata.  

---
### Quick Start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run cells **1 → 20** top to bottom (Cell 6 will ask you to mount Drive — do it)
3. Paste your free **ngrok auth token** into Cell 5's `NGROK_AUTH_TOKEN` field
4. After Cell 20 finishes, **run Cell 21** to test the live server

> ⚠️ **Do not run a training loop from this notebook.** This is inference-only using pretrained weights.

## Section 2 — Environment & GPU Check

This cell **must pass** before anything else. If it raises `RuntimeError`, go to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU -> Save, then reconnect."
    )
print(result.stdout)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("torch.cuda.is_available() returned False even though nvidia-smi passed. Try restarting the runtime.")

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"✅ VRAM: {total_vram:.1f} GB total")
print(f"✅ PyTorch version: {torch.__version__}")

## Section 3 — Dependency Installation

**Strategy:** We do NOT touch `torch` or `torchvision`. Colab pre-installs a version of both that are compiled together and ABI-matched. Overwriting either will break native C++ ops (e.g. `torchvision::nms`).  
Instead we install `xformers` without a version pin so pip selects the wheel compiled for the already-installed torch.  
All other packages are pinned to their latest Colab-compatible versions.  
**This cell takes ~2-4 minutes on first run.**

In [ ]:
import subprocess, sys

# ──────────────────────────────────────────────────────────────────────────────
# IMPORTANT: Do NOT add torch / torchvision / torchaudio here.
# Colab pre-installs a mutually ABI-compatible set. Overwriting any one of them
# breaks native C++ ops (torchvision::nms, etc.) and causes runtime crashes.
# xformers is intentionally left unpinned so pip selects the wheel that was
# compiled against the exact torch version already installed in this runtime.
# ──────────────────────────────────────────────────────────────────────────────

packages = [
    "diffusers==0.31.0",
    "transformers==4.46.1",
    "accelerate==1.0.1",
    "controlnet-aux==0.0.9",
    "xformers",          # no version pin — pip auto-selects wheel for installed torch
    "fastapi==0.115.2",
    "uvicorn==0.34.0",
    "pyngrok==7.2.0",
    "python-multipart==0.0.18",
    "nest-asyncio==1.6.0",
    "pillow",
    "numpy",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("pip install failed — see stderr above.")

print("Installation complete.")

# Sanity-check: confirm torch/torchvision are still ABI-matched
import torch, torchvision
try:
    # If this import works, the C++ ops are intact.
    from torchvision.ops import nms as _nms_check
    print(f"✅ torch=={torch.__version__}  torchvision=={torchvision.__version__}  — C++ ops intact")
except Exception as e:
    raise RuntimeError(
        f"torchvision C++ op broken after install: {e}\n"
        "Go to Runtime → Factory reset runtime, then re-run from Section 2."
    )

In [ ]:
import diffusers, transformers, accelerate, fastapi, uvicorn, pyngrok, PIL, numpy, nest_asyncio
import importlib.metadata

# For packages with no __version__, fall back to importlib.metadata
def get_ver(mod, name):
    v = getattr(mod, '__version__', None)
    if v is None:
        try: v = importlib.metadata.version(name)
        except Exception: v = 'unknown'
    return v

versions = {
    "diffusers":         (get_ver(diffusers, 'diffusers'),          "0.31.0"),
    "transformers":      (get_ver(transformers, 'transformers'),     "4.46.1"),
    "accelerate":        (get_ver(accelerate, 'accelerate'),         "1.0.1"),
    "fastapi":           (get_ver(fastapi, 'fastapi'),               "0.115.2"),
    "uvicorn":           (get_ver(uvicorn, 'uvicorn'),               "0.34.0"),
    "pyngrok":           (get_ver(pyngrok, 'pyngrok'),               "7.2.0"),
    "pillow":            (get_ver(PIL, 'pillow'),                     None),
    "numpy":             (get_ver(numpy, 'numpy'),                   None),
    "nest_asyncio":      (get_ver(nest_asyncio, 'nest-asyncio'),     None),
}

all_ok = True
for pkg, (installed, expected) in versions.items():
    status = "✅"
    if expected and installed != expected:
        status = f"⚠️  MISMATCH (expected {expected})"
        all_ok = False
    print(f"{status}  {pkg}=={installed}")

# Also print xformers version separately (no __version__ in all builds)
try:
    xf_ver = importlib.metadata.version('xformers')
    print(f"✅  xformers=={xf_ver}  (auto-selected for installed torch)")
except Exception:
    print("⚠️  xformers not found — memory-efficient attention will fall back to standard.")

if not all_ok:
    print("\n⚠️ Version mismatches above. Compatibility is not guaranteed.")
else:
    print("\n✅ All packages verified.")

## Section 4 — Imports

In [ ]:
import io
import base64
import time
import logging
import os

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel
from transformers import CLIPProcessor, CLIPModel
from controlnet_aux import MLSDdetector

from fastapi import FastAPI, UploadFile, File, Form, HTTPException
import uvicorn
import nest_asyncio
from pyngrok import ngrok

print("✅ All imports successful.")

## Section 5 — Configuration

**All constants live here.** Nothing is hardcoded anywhere else in the notebook.  
Get a free ngrok token at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) (free signup, no credit card).

In [ ]:
IMG_SIZE = (512, 512)
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.5
STRENGTH = 0.75
CONTROLNET_CONDITIONING_SCALE = 1.0
SEEDS = [42, 123, 777]  # MUST NOT CHANGE — backend history/regenerate feature depends on these exact values

ROOM_LABELS = ["bedroom", "living_room", "kitchen", "office", "dining_room"]
CLIP_LABEL_PROMPTS = {
    "bedroom": "a photo of a bedroom",
    "living_room": "a photo of a living room",
    "kitchen": "a photo of a kitchen",
    "office": "a photo of an office",
    "dining_room": "a photo of a dining room",
}

DRIVE_CACHE_DIR = "/content/drive/MyDrive/roomcanvas_model_cache"
NGROK_AUTH_TOKEN = "" #@param {type:"string"}

print("✅ Configuration loaded.")
print(f"   Seeds: {SEEDS}  |  IMG_SIZE: {IMG_SIZE}  |  Steps: {NUM_INFERENCE_STEPS}")

## Section 6 — Model Download & Weight Caching (Google Drive)

Mounts Drive and points diffusers at a persistent cache. **First run:** ~5 GB download (several minutes). **Subsequent runs:** loads from Drive cache in ~30 seconds.  
If you skip Drive mounting, weights download to `/content` — this works but does NOT persist across disconnections.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    cache_dir = DRIVE_CACHE_DIR
    print(f"✅ Drive mounted. Model cache: {cache_dir}")
except Exception as e:
    cache_dir = "/content/model_cache"
    os.makedirs(cache_dir, exist_ok=True)
    print(f"⚠️  Drive mount failed or skipped: {e}")
    print(f"   Using local cache (will not persist): {cache_dir}")

## Section 7 — Model Loading & Pipeline Initialization

In [ ]:
print("Loading ControlNet (MLSD)...")
try:
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-mlsd",
        torch_dtype=torch.float16,
        cache_dir=cache_dir,
    )
except OSError as e:
    raise RuntimeError(f"Network/download error loading ControlNet: {e}. Check internet or Drive cache.") from e
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError("CUDA OOM loading ControlNet. Try Runtime -> Factory reset runtime.") from e

print("Loading SD 1.5 + ControlNet pipeline...")
try:
    pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        torch_dtype=torch.float16,
        safety_checker=None,
        cache_dir=cache_dir,
    ).to("cuda")
except OSError as e:
    raise RuntimeError(f"Network/download error loading SD1.5: {e}.") from e
except ImportError as e:
    raise RuntimeError(f"Missing dependency: {e}. Re-run Section 3.") from e
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError("CUDA OOM loading SD1.5. Try Runtime -> Factory reset runtime.") from e

print("✅ Pipeline loaded.")

## Section 8 — Memory Optimization Setup

In [ ]:
try:
    pipe.enable_xformers_memory_efficient_attention()
    print("✅ xformers memory-efficient attention enabled.")
except Exception as e:
    print(f"⚠️  xformers not available ({e}). Using standard attention.")

pipe.enable_vae_slicing()
print("✅ VAE slicing enabled.")
pipe.enable_attention_slicing()
print("✅ Attention slicing enabled.")

allocated = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nVRAM after model load: {allocated:.2f} / {total:.1f} GB")

## Section 9 — Helper: Image Preprocessing

In [ ]:
def preprocess_image(image: Image.Image, size: tuple = IMG_SIZE) -> Image.Image:
    """Convert to RGB and resize to target resolution (default 512x512)."""
    return image.convert("RGB").resize(size, Image.LANCZOS)

print("✅ preprocess_image() defined.")

## Section 10 — Helper: Room Classification (CLIP Zero-Shot)

In [ ]:
print("Loading CLIP model...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", cache_dir=cache_dir).to("cuda")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", cache_dir=cache_dir)
print("✅ CLIP loaded.")


def classify_room(image: Image.Image) -> tuple:
    """
    Zero-shot room classification. Returns (room_type, confidence) where
    room_type is one of ROOM_LABELS and confidence is a float in [0, 1].
    """
    labels = list(CLIP_LABEL_PROMPTS.keys())
    prompts = list(CLIP_LABEL_PROMPTS.values())
    inputs = clip_processor(text=prompts, images=image, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)[0]
    best_idx = int(probs.argmax().item())
    return labels[best_idx], float(probs[best_idx])

print("✅ classify_room() defined.")

## Section 11 — Helper: Structure Extraction (MLSD)

In [ ]:
print("Loading MLSD detector...")
mlsd_detector = MLSDdetector.from_pretrained("lllyasviel/ControlNet", cache_dir=cache_dir)
print("✅ MLSD detector loaded.")


def extract_structure(image: Image.Image) -> Image.Image:
    """Extracts MLSD structural line map (walls/windows/doors) from a room photo."""
    return mlsd_detector(image)

print("✅ extract_structure() defined.")

## Section 12 — Helper: Prompt Builder

**Verbatim port from `backend/app/services/style_templates.py` and `backend/app/services/prompt_builder.py`.**  
Do NOT paraphrase or improve the prompt text — identical (room_type, style) pairs must produce byte-for-byte identical prompts in the backend and this notebook.

In [ ]:
# ---------------------------------------------------------------------------
# Verbatim port of backend/app/services/style_templates.py
# ---------------------------------------------------------------------------
STYLE_TEMPLATES = {
    "modern_minimalist": {
        "furniture": [
            "Low-profile platform sofa with hidden supports",
            "Monolithic concrete or stone coffee table",
            "Handleless matte-finish storage cabinets",
            "Recessed track lighting with slim spotlights"
        ],
        "palette": ["Pure White", "Warm Charcoal", "Soft Beige", "Matte Black"],
        "budget_tag": "Mid-Range",
        "reason_template": "Focuses on clean lines, simplicity, and functionality by eliminating unnecessary clutter, creating a calm and open atmosphere in your {room_type} while using a monochromatic backdrop."
    },
    "scandinavian": {
        "furniture": [
            "Light oak tapered-leg side tables",
            "Woven wool lounge chair in cream or soft beige",
            "Floating wood shelves with concealed mounting",
            "Minimalist opal glass pendant lighting"
        ],
        "palette": ["Alabaster White", "Pale Ash Oak", "Mist Grey", "Sage Green accents"],
        "budget_tag": "Budget-Friendly",
        "reason_template": "Emphasizes natural light, organic textures, and functional simplicity. The warm wood tones and soft accents introduce a cozy feeling of 'hygge' into your {room_type} without sacrificing space."
    },
    "industrial": {
        "furniture": [
            "Distressed leather clean-lined sofa",
            "Reclaimed wood and steel coffee table",
            "Black iron piping wall shelving units",
            "Exposed filament Edison bulb light fixtures"
        ],
        "palette": ["Raw Iron Black", "Brick Red", "Weathered Grey", "Copper accents"],
        "budget_tag": "Mid-Range",
        "reason_template": "Celebrates raw materials and architectural features like structural steel and exposed piping, giving your {room_type} a bold, warehouse-like aesthetic filled with texture and history."
    },
    "bohemian": {
        "furniture": [
            "Rattan or wicker peacock accent chair",
            "Plush low-to-the-ground floor pillows and poufs",
            "Layered distressed vintage Persian rugs",
            "Macrame wall hangings and hanging plant holders"
        ],
        "palette": ["Warm Terracotta", "Mustard Yellow", "Olive Green", "Burnt Orange"],
        "budget_tag": "Budget-Friendly",
        "reason_template": "Embraces a carefree, layered, and eclectically styled environment. It integrates rich textiles, global patterns, and vibrant plant life, creating a highly personal and warm {room_type}."
    },
    "luxury_contemporary": {
        "furniture": [
            "Velvet upholstered custom sofa with brass plinth",
            "Polished Calacatta marble console table",
            "Bespoke geometric crystal chandelier",
            "Polished gold or brass metal frame accent chairs"
        ],
        "palette": ["Champagne Gold", "Rich Emerald", "Pearl White", "Deep Obsidian"],
        "budget_tag": "Premium",
        "reason_template": "Merges high-end materials, sophisticated textures, and custom-made statements. It delivers an elegant, refined aesthetic that exudes luxury while preserving the comfort and utility of your {room_type}."
    }
}


# ---------------------------------------------------------------------------
# Verbatim port of backend/app/services/prompt_builder.py
# ---------------------------------------------------------------------------
def build_prompt(room_type: str, style: str) -> tuple:
    """
    Builds (prompt, negative_prompt) from room_type + style.
    Byte-for-byte identical output to backend/app/services/prompt_builder.py.
    """
    style_key = style.lower().strip().replace(" ", "_")
    room_name = room_type.lower().strip().replace("_", " ")

    template = STYLE_TEMPLATES.get(style_key)
    if not template:
        template = STYLE_TEMPLATES["modern_minimalist"]
        style_key = "modern_minimalist"

    palette_str = ", ".join(template["palette"])
    furniture_str = ", ".join(template["furniture"])

    prompt = (
        f"A professionally designed {room_name} in {style_key.replace('_', ' ').title()} style. "
        f"Featuring: {furniture_str}. "
        f"Color palette: {palette_str}. "
        f"Interior design photography, realistic shadows, high-end materials, 8k resolution, "
        f"architectural digest, daytime soft lighting, clean composition."
    )

    negative_prompt = (
        "low quality, blurry, worst quality, deformed, distorted, unrealistic proportions, "
        "cluttered, messy, extra walls, missing windows, out of frame, watermark, text, signature"
    )

    return prompt, negative_prompt


# Sanity check
test_p, _ = build_prompt("living_room", "scandinavian")
assert "Scandinavian" in test_p or "scandinavian" in test_p
print("✅ STYLE_TEMPLATES and build_prompt() defined and verified.")
print(f"   Sample: {test_p[:100]}...")

## Section 13 — Core Inference Function

**Seed reproducibility is mandatory.** Seeds `[42, 123, 777]` must always produce the same images for a given prompt — the backend's history/regenerate feature depends on this.  

**Implementation note:** `diffusers==0.31.0` supports `generator=List[torch.Generator]` matched 1:1 with a list of prompts for reproducible per-seed batching. If you ever downgrade below that threshold, replace the single batched call with 3 sequential single-image calls (one per seed) with `torch.cuda.empty_cache()` between them. **Do not** fall back to `num_images_per_prompt=3` with a single seed — that produces non-reproducible variation ordering.

In [ ]:
def generate_variations(
    original_image: Image.Image,
    control_image: Image.Image,
    prompt: str,
    negative_prompt: str,
) -> tuple:
    """
    Batched diffusion call producing len(SEEDS) variations.
    Returns (variations, elapsed_seconds) where each variation dict is
    {"image": PIL.Image, "seed": int}.
    """
    start = time.time()

    generators = [torch.Generator("cuda").manual_seed(s) for s in SEEDS]

    result = pipe(
        prompt=[prompt] * len(SEEDS),
        negative_prompt=[negative_prompt] * len(SEEDS),
        image=original_image,
        control_image=control_image,
        strength=STRENGTH,
        guidance_scale=GUIDANCE_SCALE,
        controlnet_conditioning_scale=CONTROLNET_CONDITIONING_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        num_images_per_prompt=1,
        generator=generators,
    )

    elapsed = time.time() - start
    variations = [{"image": img, "seed": seed} for img, seed in zip(result.images, SEEDS)]
    return variations, elapsed

print("✅ generate_variations() defined.")

## Section 14 — Metadata & Timing Assembly

In [ ]:
def pil_to_b64(img: Image.Image) -> str:
    """Encode PIL image to base64 PNG string."""
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def assemble_response(room_type, confidence, style, prompt, control_image, variations, gen_time, total_time):
    """
    Builds the exact JSON response schema required by the integration contract.
    Supports all three backend service interfaces (ClipService, MlsdService, InferenceService)
    from a single cached response.
    """
    return {
        "room_type": room_type,
        "room_confidence": round(confidence, 4),
        "style": style,
        "prompt_used": prompt,
        "model_used": "sd1.5+controlnet-mlsd",
        "control_image_b64": pil_to_b64(control_image),
        "generation_time_sec": round(gen_time, 2),
        "total_time_sec": round(total_time, 2),
        "variations": [{"seed": v["seed"], "image_b64": pil_to_b64(v["image"])} for v in variations],
    }

print("✅ pil_to_b64() and assemble_response() defined.")

## Section 15 — Visualization (Sanity-Check Plot)

Run this offline sanity check **before** starting the server to catch OOM or model errors early.

In [ ]:
# Create a neutral grey stand-in room image for the sanity check
sanity_pil = preprocess_image(Image.fromarray(np.full((512, 512, 3), 128, dtype=np.uint8)))
print("Running sanity check (modern_minimalist style)...")

try:
    ctrl_img = extract_structure(sanity_pil)
    room_t, conf = classify_room(sanity_pil)
    prompt_s, neg_s = build_prompt(room_t, "modern_minimalist")
    vars_s, elapsed_s = generate_variations(sanity_pil, ctrl_img, prompt_s, neg_s)

    fig = plt.figure(figsize=(18, 5))
    fig.suptitle(f"Sanity Check — {room_t} ({conf:.1%}) | modern_minimalist | {elapsed_s:.1f}s", fontsize=12)
    gs = gridspec.GridSpec(1, 5, figure=fig)

    ax0 = fig.add_subplot(gs[0]); ax0.imshow(sanity_pil); ax0.set_title("Original"); ax0.axis("off")
    ax1 = fig.add_subplot(gs[1]); ax1.imshow(ctrl_img); ax1.set_title("MLSD"); ax1.axis("off")
    for i, v in enumerate(vars_s):
        ax = fig.add_subplot(gs[2+i]); ax.imshow(v["image"]); ax.set_title(f"seed={v['seed']}"); ax.axis("off")

    plt.tight_layout()
    plt.savefig("/content/sanity_check_output.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"✅ Sanity check passed in {elapsed_s:.2f}s. Output saved to /content/sanity_check_output.png")

except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    print("OOM during sanity check. Go to Runtime -> Factory reset runtime and re-run from Section 2.")
except Exception as e:
    print(f"Sanity check failed: {e}")
    raise

## Section 16 — FastAPI App Definition

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("roomcanvas")

app = FastAPI(
    title="RoomCanvas Inference Server",
    description="SD1.5 + ControlNet-MLSD inference endpoint for RoomCanvas AI.",
    version="1.0.0",
)

print("✅ FastAPI app defined.")

## Section 17 — Ngrok Tunnel Setup

**Get your free ngrok auth token:**
1. Sign up at [ngrok.com](https://ngrok.com/) (free, no credit card required)
2. Go to **Your Authtoken** in the ngrok dashboard
3. Copy the token and paste it into Section 5's `NGROK_AUTH_TOKEN` field, then re-run Section 5

The public URL below **changes every restart** — paste the new URL into `backend/.env` as `COLAB_INFERENCE_URL` each session.

In [ ]:
if not NGROK_AUTH_TOKEN:
    raise ValueError(
        "NGROK_AUTH_TOKEN is empty. Paste your free token into Section 5's NGROK_AUTH_TOKEN field "
        "and re-run Section 5 before running this cell."
    )

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# .public_url extracts the plain string from the NgrokTunnel object.
# Without this, str(tunnel) returns the full repr and causes InvalidSchema errors in requests.
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url   # e.g. 'https://xxxx.ngrok-free.app'

print(f"\n🌐 Public URL: {public_url}")
print("   Paste this into backend/.env as COLAB_INFERENCE_URL")
print("   ⚠️  This URL changes every notebook restart — update it each session.")

## Section 18 — Health Check Endpoint

In [ ]:
@app.get("/health")
def health():
    """Returns GPU name and VRAM usage. Call this to confirm the server is alive."""
    return {
        "status": "ok",
        "model": "sd1.5+controlnet-mlsd",
        "gpu": torch.cuda.get_device_name(0),
        "vram_allocated_mb": round(torch.cuda.memory_allocated() / 1024**2, 1),
        "vram_total_mb": round(torch.cuda.get_device_properties(0).total_memory / 1024**2, 1),
    }

print("✅ GET /health defined.")

## Section 19 — Main /infer Endpoint

In [ ]:
@app.post("/infer")
async def infer(image: UploadFile = File(...), style: str = Form(...)):
    """
    Full pipeline: image validation -> structure extraction -> room classification
    -> prompt building -> batched 3-variation generation -> response.

    Response schema:
    {
      room_type, room_confidence, style, prompt_used, model_used,
      control_image_b64, generation_time_sec, total_time_sec,
      variations: [{seed, image_b64}, ...] (3 items, seeds 42/123/777)
    }
    """
    request_start = time.time()

    # Validate style key
    if style not in STYLE_TEMPLATES:
        raise HTTPException(
            status_code=400,
            detail=f"Unknown style '{style}'. Valid keys: {list(STYLE_TEMPLATES.keys())}"
        )

    # Validate + load image
    try:
        contents = await image.read()
        pil_image = Image.open(io.BytesIO(contents)).convert("RGB").resize(IMG_SIZE)
    except Exception as exc:
        logger.error(f"Image load failed: {exc}")
        raise HTTPException(status_code=400, detail=f"Invalid image file: {exc}")

    # Stage 1: structure extraction
    t0 = time.time()
    try:
        control_image = extract_structure(pil_image)
    except Exception as exc:
        logger.error(f"MLSD extraction failed: {exc}")
        raise HTTPException(status_code=500, detail="Structure extraction failed")
    mlsd_time = round(time.time() - t0, 2)

    # Stage 2: room classification
    t0 = time.time()
    try:
        room_type, confidence = classify_room(pil_image)
    except Exception as exc:
        logger.error(f"CLIP classification failed: {exc}")
        raise HTTPException(status_code=500, detail="Room classification failed")
    clip_time = round(time.time() - t0, 2)

    # Stage 3: prompt building
    prompt, negative_prompt = build_prompt(room_type, style)

    # Stage 4: batched generation (with one OOM retry)
    try:
        variations, gen_time = generate_variations(pil_image, control_image, prompt, negative_prompt)
    except torch.cuda.OutOfMemoryError:
        logger.warning("CUDA OOM -- clearing cache and retrying once")
        torch.cuda.empty_cache()
        try:
            variations, gen_time = generate_variations(pil_image, control_image, prompt, negative_prompt)
        except Exception as exc:
            logger.error(f"Inference failed after OOM retry: {exc}")
            raise HTTPException(status_code=500, detail="Inference failed after OOM retry")
    except Exception as exc:
        logger.error(f"Inference failed: {exc}")
        raise HTTPException(status_code=500, detail=f"Inference failed: {exc}")

    total_time = time.time() - request_start
    logger.info(
        f"room={room_type} ({confidence:.1%}) style={style} "
        f"mlsd={mlsd_time}s clip={clip_time}s gen={gen_time:.2f}s total={total_time:.2f}s"
    )

    return assemble_response(
        room_type=room_type, confidence=confidence, style=style, prompt=prompt,
        control_image=control_image, variations=variations, gen_time=gen_time, total_time=total_time,
    )

print("✅ POST /infer defined.")

## Section 20 — Server Startup

> ⚠️ **This server runs in a background thread.**  
> This cell will finish immediately, but the server will stay alive in the background.  
> You can safely proceed to run Section 21 in the same notebook without it queuing up!

In [ ]:
import threading
import time

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

# Start uvicorn in a background thread so it doesn't block Colab
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print(f"🚀 Starting server on port 8000 (in background)")
print(f"   Health: {public_url}/health")
print(f"   Infer:  POST {public_url}/infer\n")
print("✅ Server is running in the background. You can now run Section 21.")
time.sleep(2)  # Give uvicorn a moment to bind the port before continuing

## Section 21 — Live Test Cell

**Run this in the same notebook after Section 20.**  
Tests all 5 valid styles, one invalid style (expects HTTP 400), and a corrupt upload (expects HTTP 400).

In [ ]:
import requests
from google.colab import files

test_img_path = "/content/test_room.jpg"
if not os.path.exists(test_img_path):
    print("Upload a room photo (JPG/PNG):")
    uploaded = files.upload()
    if uploaded:
        test_img_path = list(uploaded.keys())[0]
    else:
        print("No file uploaded — using neutral grey stand-in.")
        Image.fromarray(np.full((512, 512, 3), 180, dtype=np.uint8)).save(test_img_path)

STYLES_TO_TEST = ["modern_minimalist", "scandinavian", "industrial", "bohemian", "luxury_contemporary"]
SERVER_URL = str(public_url)

test_results = []
print(f"Testing {len(STYLES_TO_TEST)} styles against {SERVER_URL}/infer...\n")

for style_name in STYLES_TO_TEST:
    print(f"  -> {style_name} ... ", end="", flush=True)
    try:
        with open(test_img_path, "rb") as f:
            resp = requests.post(
                f"{SERVER_URL}/infer",
                files={"image": ("room.jpg", f, "image/jpeg")},
                data={"style": style_name},
                timeout=120,
            )
        if resp.status_code == 200:
            data = resp.json()
            test_results.append((style_name, data))
            print(f"OK  room={data['room_type']} ({data['room_confidence']:.1%}) total={data['total_time_sec']}s gen={data['generation_time_sec']}s")
        else:
            print(f"FAIL HTTP {resp.status_code}: {resp.text[:200]}")
    except Exception as e:
        print(f"ERROR: {e}")

for style_name, data in test_results:
    vars_data = data["variations"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"{style_name} | {data['room_type']} ({data['room_confidence']:.1%}) | {data['total_time_sec']}s total", fontsize=11)
    for ax, v in zip(axes, vars_data):
        pil_v = Image.open(io.BytesIO(base64.b64decode(v["image_b64"])))
        ax.imshow(pil_v); ax.set_title(f"Seed {v['seed']}"); ax.axis("off")
    plt.tight_layout()
    plt.show()

# Error case tests
print("\nTesting invalid style key (expect HTTP 400)...")
with open(test_img_path, "rb") as f:
    bad_style_resp = requests.post(f"{SERVER_URL}/infer", files={"image": ("r.jpg", f, "image/jpeg")}, data={"style": "not_valid"}, timeout=10)
assert bad_style_resp.status_code == 400, f"Expected 400 got {bad_style_resp.status_code}"
print(f"OK HTTP 400: {bad_style_resp.json()}")

print("Testing corrupt file upload (expect HTTP 400)...")
bad_file_resp = requests.post(f"{SERVER_URL}/infer", files={"image": ("bad.jpg", b"not an image", "image/jpeg")}, data={"style": "scandinavian"}, timeout=10)
assert bad_file_resp.status_code == 400, f"Expected 400 got {bad_file_resp.status_code}"
print(f"OK HTTP 400: {bad_file_resp.json()}")

print(f"\n✅ Live test complete. {len(test_results)}/{len(STYLES_TO_TEST)} styles passed.")

## Section 22 — Performance Diagnostics

In [ ]:
vram_alloc  = torch.cuda.memory_allocated() / 1024**3
vram_res    = torch.cuda.memory_reserved() / 1024**3
vram_total  = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("━" * 52)
print("  VRAM Diagnostics")
print("━" * 52)
print(f"  Allocated : {vram_alloc:.2f} GB")
print(f"  Reserved  : {vram_res:.2f} GB")
print(f"  Total     : {vram_total:.1f} GB")
print(f"  Free      : {vram_total - vram_res:.2f} GB")

if test_results:
    total_times = [d["total_time_sec"] for _, d in test_results]
    gen_times   = [d["generation_time_sec"] for _, d in test_results]
    avg_total   = sum(total_times) / len(total_times)
    avg_gen     = sum(gen_times) / len(gen_times)

    print()
    print("━" * 52)
    print("  Timing Breakdown (from live test)")
    print("━" * 52)
    for style_name, d in test_results:
        print(f"  {style_name:25s}  total={d['total_time_sec']:5.1f}s  gen={d['generation_time_sec']:5.1f}s")
    print(f"  {'AVERAGE':25s}  total={avg_total:5.1f}s  gen={avg_gen:5.1f}s")

    print()
    THRESHOLD = 60
    if avg_total <= THRESHOLD:
        print(f"  ✅ Average total request time: {avg_total:.1f}s — within acceptable demo range (<{THRESHOLD}s)")
    else:
        print(f"  ⚠️  Average total request time: {avg_total:.1f}s — EXCEEDS {THRESHOLD}s threshold.")
        print("      Consider reducing NUM_INFERENCE_STEPS to 20 in Section 5.")
else:
    print("\n  ⚠️  No test_results — run Section 21 first.")

## Section 23 — Cleanup (Optional)

Run only when done. Restart the runtime before starting a new session.

In [ ]:
try:
    ngrok.disconnect(public_url)
    print("✅ ngrok tunnel disconnected.")
except Exception as e:
    print(f"⚠️  ngrok disconnect: {e}")

try:
    del pipe, controlnet, clip_model, mlsd_detector
    torch.cuda.empty_cache()
    print("✅ Models deleted and CUDA cache cleared.")
except Exception as e:
    print(f"⚠️  Cleanup: {e}")

print("\nResources released. Restart runtime before next session for a clean state.")

## Section 24 — Troubleshooting

| Symptom | Cause | Fix |
|---------|-------|-----|
| `RuntimeError: No GPU detected` in Section 2 | CPU-only runtime | Runtime → Change runtime type → **T4 GPU** → Save → Reconnect |
| `CUDA out of memory` during generation | T4 under VRAM pressure | Runtime → Factory reset → re-run from Section 2. If persistent: reduce `NUM_INFERENCE_STEPS` to 20 or switch to 3 sequential single-image calls (see Section 13 note) |
| ngrok returns `ERR_NGROK_3200` | Invalid/expired auth token | Get fresh token at [dashboard.ngrok.com](https://dashboard.ngrok.com) → paste into Section 5 → re-run Sections 5 and 17 |
| MLSD output is blank white | `controlnet_aux==0.0.9` not installed | Re-run Section 3. Verify `import controlnet_aux; print(controlnet_aux.__version__)` prints `0.0.9` |
| CLIP always predicts same room type | Label prompts too short or grey test image | Use a real room photo. Verify Section 5's `CLIP_LABEL_PROMPTS` are full sentences like `"a photo of a bedroom"` |
| Colab disconnects mid-session | Free tier idle/max-runtime limits (~90 min / ~12 h) | Reconnect and re-run only Sections 6–8 then Section 20 — weights reload from Drive cache in ~30s |
| `/infer` returns 400 for a valid style | Case or underscore mismatch | Valid keys (exact): `modern_minimalist`, `scandinavian`, `industrial`, `bohemian`, `luxury_contemporary` |
